# OpenCV Cascade Classifier Tutorial: From 0 to Mastery

This notebook follows the style of the previous OpenCV series notebooks: theory first, runnable local demos second, then debugging rules, practice tasks, and references.

本 notebook 用“从 0 到精通”的路线讲清楚 Cascade Classifier / 级联分类器。你会学到：

- Cascade Classifier 到底解决什么问题
- Haar-like features / Haar 特征为什么能描述人脸、眼睛等结构
- Integral image / 积分图如何让矩形区域求和变快
- AdaBoost 如何把很多弱分类器组合成强分类器
- Cascade / 级联结构为什么能快速拒绝大多数背景窗口
- `cv2.CascadeClassifier.detectMultiScale` 的参数如何影响结果
- 为什么会漏检、误检，以及真实项目里应该如何调试

Core mindset:

`cascade detection = scan many windows + reject easy negatives early + verify harder candidates later`

## How to Use This Notebook

Recommended learning order:

1. Read Part 1 to build the mental model.
2. Run Part 2 cell by cell and inspect every visualization.
3. Modify scale, window size, thresholds, and `detectMultiScale` parameters.
4. Use Part 3 as a debugging checklist and project roadmap.

Suggested environment:

- Python 3.10+
- `opencv-python`
- `numpy`
- `matplotlib`

This notebook uses OpenCV's built-in cascade XML files when available. The synthetic demos run without downloading external images.

## Part 1: Theoretical Foundations

### 1.1 What Problem Is a Cascade Classifier Solving?

A detector receives an image and asks:

`Does an object of a known class exist somewhere in this image? If yes, where?`

Classic cascade classifiers are usually used for rigid or semi-rigid objects with repeatable appearance:

- frontal faces
- eyes
- smiles
- license plates
- simple industrial parts

The basic detection pipeline:

1. Convert image to grayscale.
2. Scan many windows over many scales.
3. For each window, compute simple rectangle contrast features.
4. Pass the window through a cascade of stages.
5. Reject it as soon as one stage says no.
6. Keep windows that pass all stages.
7. Merge overlapping detections.

The key idea:

`Most windows are background. A good detector should spend almost no time on obvious background.`

### 1.2 Haar-Like Features

A Haar-like feature compares sums of pixel intensities in adjacent rectangles.

Examples:

- eye region may be darker than upper cheeks
- nose bridge may be brighter than nearby shadows
- mouth region may create a dark horizontal band
- object edges create strong left-right or top-bottom contrast

A feature is usually a weighted sum:

`feature_value = sum(white rectangles) - sum(black rectangles)`

One Haar feature is weak. Thousands of selected features, arranged carefully, become useful.

Practical rule:

`Cascade classifiers do not understand objects semantically. They detect learned contrast patterns.`

### 1.3 Integral Image: Fast Rectangle Sums

If every window needs many rectangle sums, naive summation is too slow.

An integral image stores cumulative sums from the top-left corner:

`integral(x, y) = sum of all pixels above and left of (x, y)`

Then any rectangle sum can be computed with four lookups:

`rect_sum = D - B - C + A`

This is why old-school Haar cascades could run in real time on modest hardware.

### 1.4 AdaBoost and Cascade Stages

Training a cascade has two major ideas.

**AdaBoost** selects useful weak classifiers:

- each weak classifier tests one feature against a threshold
- weak classifiers are weighted and combined
- training focuses more on examples that previous weak classifiers got wrong

**Cascade stages** make detection fast:

- early stages are cheap and reject obvious non-objects
- later stages are more detailed and only see surviving candidates
- a candidate must pass every stage to become a detection

OpenCV mostly exposes the trained detector for inference. Training your own cascade is possible, but the common workflow is using pre-trained XML files.

### 1.5 Failure Modes

Cascade classifiers often fail when:

- the object pose differs from training data
- the object is too small or too large for the search settings
- lighting changes the learned contrast pattern
- blur removes edge and texture evidence
- partial occlusion hides important regions
- background accidentally matches the learned rectangle contrasts
- `scaleFactor`, `minNeighbors`, or `minSize` are poorly chosen
- the cascade XML file is for a different object or pose

Master-level habit:

`Always debug the search space: grayscale quality, object scale, candidate count, and grouping threshold.`

## Part 2: Coding Demo - Build the Intuition Step by Step

### 2.1 Imports and Helpers

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

try:
    import cv2
except ImportError as exc:
    raise ImportError("OpenCV is not installed. Please run: pip install opencv-python") from exc

print("OpenCV version:", cv2.__version__)
print("Working directory:", os.getcwd())
print("Cascade data directory:", cv2.data.haarcascades)


def show_many(images, cols=3, figsize=(15, 8)):
    rows = int(np.ceil(len(images) / cols))
    plt.figure(figsize=figsize)
    for idx, (title, image, cmap) in enumerate(images, start=1):
        plt.subplot(rows, cols, idx)
        if image.ndim == 2:
            plt.imshow(image, cmap=cmap or "gray")
        else:
            plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        plt.title(title)
        plt.axis("off")
    plt.tight_layout()
    plt.show()


def draw_boxes(frame, boxes, color=(0, 255, 0), label=None):
    vis = frame.copy()
    for i, (x, y, w, h) in enumerate(np.asarray(boxes).reshape(-1, 4) if len(boxes) else []):
        x, y, w, h = [int(v) for v in (x, y, w, h)]
        cv2.rectangle(vis, (x, y), (x + w, y + h), color, 2)
        if label:
            cv2.putText(vis, f"{label} {i}", (x, max(20, y - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
    return vis


def normalize_for_display(image):
    image = image.astype(np.float32)
    return cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

### 2.2 Integral Image by Hand

OpenCV's integral image has one extra top row and one extra left column. That padding makes rectangle sums easy and avoids boundary special cases.

In [ ]:
toy = np.array(
    [[3, 1, 2, 4, 1],
     [0, 2, 5, 1, 3],
     [4, 1, 1, 0, 2],
     [2, 3, 0, 2, 1]],
    dtype=np.uint8,
)
integral = cv2.integral(toy)


def integral_rect_sum(integral_image, x, y, w, h):
    x0, y0 = x, y
    x1, y1 = x + w, y + h
    return int(integral_image[y1, x1] - integral_image[y0, x1] - integral_image[y1, x0] + integral_image[y0, x0])


x, y, w, h = 1, 1, 3, 2
direct_sum = int(toy[y:y + h, x:x + w].sum())
fast_sum = integral_rect_sum(integral, x, y, w, h)

print("Toy image:\n", toy)
print("Integral image with padding:\n", integral)
print("Direct rectangle sum:", direct_sum)
print("Integral rectangle sum:", fast_sum)

### 2.3 Visualize Haar-Like Features

The visualization below shows common rectangle contrast patterns. White rectangles add intensity; black rectangles subtract intensity.

In [ ]:
def haar_feature_canvas(kind="two_horizontal", size=120):
    canvas = np.full((size, size), 128, dtype=np.uint8)
    if kind == "two_horizontal":
        canvas[:, :size // 2] = 230
        canvas[:, size // 2:] = 30
    elif kind == "two_vertical":
        canvas[:size // 2, :] = 230
        canvas[size // 2:, :] = 30
    elif kind == "three_horizontal":
        third = size // 3
        canvas[:, :third] = 30
        canvas[:, third:2 * third] = 230
        canvas[:, 2 * third:] = 30
    elif kind == "four_checker":
        half = size // 2
        canvas[:half, :half] = 230
        canvas[half:, half:] = 230
        canvas[:half, half:] = 30
        canvas[half:, :half] = 30
    return canvas


feature_images = [
    ("Two-rectangle horizontal", haar_feature_canvas("two_horizontal"), "gray"),
    ("Two-rectangle vertical", haar_feature_canvas("two_vertical"), "gray"),
    ("Three-rectangle", haar_feature_canvas("three_horizontal"), "gray"),
    ("Four-rectangle checker", haar_feature_canvas("four_checker"), "gray"),
]
show_many(feature_images, cols=4, figsize=(14, 3))

### 2.4 Compute a Simple Haar Feature

This mini example is not a trained detector. It only shows the math behind one weak classifier: compare a bright region and a dark region.

In [ ]:
patch = np.full((80, 120), 150, dtype=np.uint8)
patch[20:45, 25:55] = 40
patch[20:45, 65:95] = 40
patch[50:65, 35:85] = 60

ii = cv2.integral(patch)
left_eye = integral_rect_sum(ii, 25, 20, 30, 25)
right_eye = integral_rect_sum(ii, 65, 20, 30, 25)
cheek = integral_rect_sum(ii, 35, 45, 50, 20)
eye_area = 30 * 25 * 2
cheek_area = 50 * 20

eye_mean = (left_eye + right_eye) / eye_area
cheek_mean = cheek / cheek_area
feature_value = cheek_mean - eye_mean

vis = cv2.cvtColor(patch, cv2.COLOR_GRAY2BGR)
cv2.rectangle(vis, (25, 20), (55, 45), (0, 0, 255), 2)
cv2.rectangle(vis, (65, 20), (95, 45), (0, 0, 255), 2)
cv2.rectangle(vis, (35, 45), (85, 65), (0, 255, 0), 2)

show_many([("Synthetic face-like patch", vis, None)], cols=1, figsize=(5, 4))
print("Eye mean:", round(eye_mean, 2))
print("Cheek mean:", round(cheek_mean, 2))
print("Feature value = cheek_mean - eye_mean:", round(feature_value, 2))

### 2.5 Simulate a Tiny Cascade

A real cascade has many learned stages. This toy version uses handcrafted thresholds so you can see the control flow: cheap tests first, expensive tests later.

In [ ]:
rng = np.random.default_rng(7)
scene = np.full((220, 320), 170, dtype=np.uint8)
scene += rng.normal(0, 8, scene.shape).astype(np.int16).clip(-20, 20).astype(np.uint8)

# Add one face-like target and several distractors.
scene[70:150, 115:205] = 155
scene[90:112, 135:158] = 35
scene[90:112, 168:191] = 35
scene[126:140, 145:178] = 55
cv2.rectangle(scene, (35, 40), (95, 100), 60, -1)
cv2.rectangle(scene, (235, 120), (295, 180), 210, -1)


def toy_cascade_score(window):
    h, w = window.shape
    top = window[:h // 2, :]
    bottom = window[h // 2:, :]
    left_eye = window[h // 4:h // 2, w // 5:2 * w // 5]
    right_eye = window[h // 4:h // 2, 3 * w // 5:4 * w // 5]
    center = window[h // 2:3 * h // 4, w // 3:2 * w // 3]

    stage1 = float(top.mean()) < float(bottom.mean()) + 25
    stage2 = float(left_eye.mean()) < 100 and float(right_eye.mean()) < 100
    stage3 = float(center.mean()) < 120
    return stage1, stage2, stage3


win_w, win_h = 90, 80
step = 10
passed_stage = {1: [], 2: [], 3: []}
for y in range(0, scene.shape[0] - win_h + 1, step):
    for x in range(0, scene.shape[1] - win_w + 1, step):
        s1, s2, s3 = toy_cascade_score(scene[y:y + win_h, x:x + win_w])
        if s1:
            passed_stage[1].append((x, y, win_w, win_h))
        if s1 and s2:
            passed_stage[2].append((x, y, win_w, win_h))
        if s1 and s2 and s3:
            passed_stage[3].append((x, y, win_w, win_h))

scene_bgr = cv2.cvtColor(scene, cv2.COLOR_GRAY2BGR)
show_many(
    [("Scene", scene_bgr, None),
     ("After stage 1", draw_boxes(scene_bgr, passed_stage[1], (255, 0, 0)), None),
     ("After stage 2", draw_boxes(scene_bgr, passed_stage[2], (0, 255, 255)), None),
     ("After stage 3", draw_boxes(scene_bgr, passed_stage[3], (0, 255, 0)), None)],
    cols=2,
    figsize=(12, 7),
)
print("Windows after stage 1:", len(passed_stage[1]))
print("Windows after stage 2:", len(passed_stage[2]))
print("Windows after stage 3:", len(passed_stage[3]))

### 2.6 Load an OpenCV Cascade XML

OpenCV ships several Haar cascade XML files. The most common first example is frontal face detection.

In [ ]:
face_cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
eye_cascade_path = cv2.data.haarcascades + "haarcascade_eye.xml"

face_cascade = cv2.CascadeClassifier(face_cascade_path)
eye_cascade = cv2.CascadeClassifier(eye_cascade_path)

if face_cascade.empty():
    raise RuntimeError(f"Could not load face cascade: {face_cascade_path}")
if eye_cascade.empty():
    raise RuntimeError(f"Could not load eye cascade: {eye_cascade_path}")

print("Loaded face cascade:", face_cascade_path)
print("Loaded eye cascade:", eye_cascade_path)

### 2.7 Run `detectMultiScale` on Your Own Image

Set `image_path` to a local photo. The cell is safe when the file does not exist: it prints a message and keeps going.

Important: Haar face cascades expect upright frontal-ish faces. Profile faces, tilted faces, tiny faces, and heavily stylized drawings often fail.

In [ ]:
image_path = "your_local_face_photo.jpg"  # Replace with a real local path if you want to test a photo.

if os.path.exists(image_path):
    image = cv2.imread(image_path)
    if image is None:
        raise RuntimeError(f"OpenCV could not read: {image_path}")
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray_eq = cv2.equalizeHist(gray)
    faces = face_cascade.detectMultiScale(
        gray_eq,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(40, 40),
    )
    result = draw_boxes(image, faces, (0, 255, 0), "face")
    show_many([("Input", image, None), ("Equalized grayscale", gray_eq, "gray"), ("Detections", result, None)], cols=3, figsize=(15, 4))
    print("Face detections:", len(faces))
else:
    print(f"No local image found at {image_path!r}. Replace image_path with a real file to run face detection.")

### 2.8 Understand `detectMultiScale` Parameters

OpenCV API:

```python
boxes = cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40))
```

Parameter intuition:

- `scaleFactor`: image pyramid step. Smaller values search more scales and run slower, but may find more objects.
- `minNeighbors`: grouping strictness. Higher values reduce false positives but can remove weak true positives.
- `minSize`: ignore objects smaller than this. Use it to reduce noise and speed up detection.
- `maxSize`: optional upper bound when you know objects cannot be huge.

Practical starting points:

- `scaleFactor=1.05` for careful search
- `scaleFactor=1.1` for normal search
- `minNeighbors=3` for more recall
- `minNeighbors=5` or `7` for fewer false positives

### 2.9 Parameter Lab Template

When you have a real local image, this cell compares several parameter settings side by side.

In [ ]:
if os.path.exists(image_path):
    image = cv2.imread(image_path)
    gray = cv2.equalizeHist(cv2.cvtColor(image, cv2.COLOR_BGR2GRAY))
    settings = [
        (1.05, 3, (30, 30)),
        (1.05, 6, (30, 30)),
        (1.10, 5, (40, 40)),
        (1.20, 5, (40, 40)),
    ]
    visuals = []
    for scale_factor, min_neighbors, min_size in settings:
        boxes = face_cascade.detectMultiScale(
            gray,
            scaleFactor=scale_factor,
            minNeighbors=min_neighbors,
            minSize=min_size,
        )
        title = f"scale={scale_factor}, neighbors={min_neighbors}, count={len(boxes)}"
        visuals.append((title, draw_boxes(image, boxes, (0, 255, 0)), None))
    show_many(visuals, cols=2, figsize=(12, 8))
else:
    print("Parameter lab skipped. Set image_path to a real local image first.")

### 2.10 Face + Eye Detection Pattern

A common cascade pattern is hierarchical detection: detect faces first, then search for eyes only inside each face ROI. This is faster and reduces false positives.

In [ ]:
if os.path.exists(image_path):
    image = cv2.imread(image_path)
    gray = cv2.equalizeHist(cv2.cvtColor(image, cv2.COLOR_BGR2GRAY))
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40))
    result = image.copy()
    for x, y, w, h in faces:
        cv2.rectangle(result, (x, y), (x + w, y + h), (0, 255, 0), 2)
        face_gray = gray[y:y + h, x:x + w]
        face_color = result[y:y + h, x:x + w]
        eyes = eye_cascade.detectMultiScale(face_gray, scaleFactor=1.1, minNeighbors=8, minSize=(15, 15))
        for ex, ey, ew, eh in eyes:
            cv2.rectangle(face_color, (ex, ey), (ex + ew, ey + eh), (255, 0, 0), 2)
    show_many([("Face and eye cascades", result, None)], cols=1, figsize=(7, 6))
    print("Faces:", len(faces))
else:
    print("Face + eye demo skipped. Set image_path to a real local image first.")

### 2.11 Template for Real Video or Webcam Detection

This cell prints a safe template instead of automatically opening your camera.

In [ ]:
webcam_cascade_template = r'''
import cv2

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
if face_cascade.empty():
    raise RuntimeError("Could not load face cascade")

cap = cv2.VideoCapture(0)  # or replace 0 with a video path
if not cap.isOpened():
    raise RuntimeError("Cannot open video source")

while True:
    ok, frame = cap.read()
    if not ok:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.equalizeHist(gray)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(40, 40),
    )

    for x, y, w, h in faces:
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

    cv2.imshow("cascade detection", frame)
    if (cv2.waitKey(30) & 0xFF) == 27:
        break

cap.release()
cv2.destroyAllWindows()
'''
print(webcam_cascade_template)

## Part 3: Debugging and Mastery Roadmap

Debug in this order:

1. **Cascade file**: verify `CascadeClassifier.empty()` is false.
2. **Grayscale quality**: inspect blur, contrast, exposure, and shadows.
3. **Object pose**: frontal cascade expects frontal objects; profile cascade expects profile objects.
4. **Object scale**: adjust `minSize`, `maxSize`, and camera distance.
5. **Search granularity**: lower `scaleFactor` if objects are missed between pyramid levels.
6. **Grouping strictness**: lower `minNeighbors` for recall; raise it for precision.
7. **ROI strategy**: search inside likely regions instead of the full image when possible.
8. **Preprocessing**: try histogram equalization, resizing, denoising, or illumination normalization.
9. **False positives**: add geometric rules, temporal smoothing, or a second-stage verifier.
10. **Runtime**: downsample frames, set `minSize`, or detect every N frames and track between detections.

Ways to improve reliability:

- Use the cascade as a proposal generator, not as absolute truth.
- Combine with tracking: detect occasionally, track between detections.
- Restrict search regions using scene knowledge.
- Add temporal voting across several frames.
- Use modern deep detectors when pose, lighting, or object variety is large.

## Part 4: Practice Projects

Suggested progression:

1. **Local photo face detector**: run the face cascade on several local photos and compare settings.
2. **Face + eye pipeline**: detect faces first, then detect eyes only inside face boxes.
3. **Parameter report**: create a table for `scaleFactor`, `minNeighbors`, count, and runtime.
4. **Webcam detector**: draw face boxes live and smooth boxes over time.
5. **Detector + tracker hybrid**: use cascade detections to initialize MeanShift, CamShift, or optical flow.
6. **False positive collector**: save crops that were wrongly detected and study their patterns.
7. **ROI speedup**: detect only in the upper half of the frame or inside a motion mask.
8. **Classical vs modern comparison**: compare Haar cascade results with a DNN detector on the same images.

## Part 5: Quick API Reference

Core OpenCV functions:

- `cv2.CascadeClassifier(path)`
- `cascade.empty()`
- `cascade.detectMultiScale(image, scaleFactor, minNeighbors, minSize, maxSize)`
- `cv2.integral(image)`
- `cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)`
- `cv2.equalizeHist(gray)`
- `cv2.rectangle(image, pt1, pt2, color, thickness)`
- `cv2.VideoCapture(source)`

Useful cascade XML files commonly shipped with OpenCV:

- `haarcascade_frontalface_default.xml`
- `haarcascade_frontalface_alt.xml`
- `haarcascade_eye.xml`
- `haarcascade_smile.xml`
- `haarcascade_profileface.xml`

Official references:

- OpenCV Cascade Classifier tutorial: https://docs.opencv.org/4.x/db/d28/tutorial_cascade_classifier.html
- OpenCV CascadeClassifier class reference: https://docs.opencv.org/4.x/d1/de5/classcv_1_1CascadeClassifier.html
- OpenCV object detection module: https://docs.opencv.org/4.x/d5/d54/group__objdetect.html

## Conclusion

Cascade Classifier teaches a classic object detection idea:

`fast detection comes from rejecting most wrong windows before doing expensive work`

Haar features provide simple contrast evidence. Integral images make those features fast. AdaBoost selects and weights useful weak tests. The cascade structure turns that collection of tests into a real-time detector. Once you understand where cascades are reliable and where they are brittle, you can use them well, debug them quickly, and combine them with tracking or modern detectors when the scene gets harder.